# Gold: Sales fact table
**Sources:** `silver.crm_sales`, `gold.dim_products`, `gold.dim_customers`  →  **Target:** `gold.fact_sales`

**What this notebook does:**
- Replace product number and customer ID with the **surrogate keys** from the dimensions
- Keep dates and numbers (sales, quantity, price)

**Run this after the two dimensions.**

In [0]:
CATALOG = "workspace"

## The business logic (SQL)
This is the **star schema**: one fact table in the middle, two dimensions around it.

In [0]:
query = f"""
SELECT
    sd.order_number,
    pr.product_key,
    cu.customer_key,
    sd.order_date,
    sd.ship_date,
    sd.due_date,
    sd.sales_amount,
    sd.quantity,
    sd.price
FROM {CATALOG}.silver.crm_sales sd
LEFT JOIN {CATALOG}.gold.dim_products pr
       ON sd.product_number = pr.product_number
LEFT JOIN {CATALOG}.gold.dim_customers cu
       ON sd.customer_id = cu.customer_id
"""
df = spark.sql(query)

## Preview

In [0]:
df.display()

## Write the Gold table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.gold.fact_sales")

## Check it Quickly

In [0]:
result = spark.table(f"{CATALOG}.gold.fact_sales")
print("rows:", result.count())